In [0]:
from pyspark.sql import functions as F

In [0]:
job_history = (

    spark.range(10000)

    .withColumn(
        "job_name",

        F.expr("""

        CASE

            WHEN rand() < .20
            THEN 'Orders_ETL'

            WHEN rand() < .40
            THEN 'Customer_ETL'

            WHEN rand() < .60
            THEN 'Inventory_Sync'

            WHEN rand() < .80
            THEN 'Sales_Aggregation'

            ELSE 'Payment_Processing'

        END

        """)
    )

    .withColumn(
        "run_id",
        F.monotonically_increasing_id()
    )

    .withColumn(
        "status",

        F.expr("""

        CASE

            WHEN rand() < .85
            THEN 'SUCCESS'

            ELSE 'FAILED'

        END

        """)
    )

    .withColumn(
        "duration_minutes",
        (F.rand()*60).cast("int")
    )

)

In [0]:
job_history = (

    job_history

    .withColumn(
        "start_time",

        F.current_timestamp()

        - F.expr(
            "INTERVAL CAST(rand()*30 AS INT) DAYS"
        )
    )

)

In [0]:
job_history = (

    job_history

    .withColumn(

        "end_time",

        F.col("start_time")

        + F.expr(
            "INTERVAL 1 HOUR"
        )
    )

)

In [0]:
job_history = (

    job_history

    .withColumn(

        "status",

        F.when(

            F.col("job_name")
            == "Inventory_Sync",

            F.when(
                F.rand() < .35,
                "FAILED"
            ).otherwise(
                F.col("status")
            )
        )

        .otherwise(
            F.col("status")
        )
    )
)

In [0]:
job_history.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"platform_monitoring.job_history"
)

In [0]:
spark.table(
"platform_monitoring.job_history"
).show(
20,
False
)

+----+-----------------+-----------+-------+----------------+--------------------------+--------------------------+
|id  |job_name         |run_id     |status |duration_minutes|start_time                |end_time                  |
+----+-----------------+-----------+-------+----------------+--------------------------+--------------------------+
|8750|Sales_Aggregation|60129542144|SUCCESS|55              |2026-06-11 01:17:35.008398|2026-06-11 02:17:35.008398|
|8751|Inventory_Sync   |60129542145|SUCCESS|37              |2026-06-01 01:17:35.008398|2026-06-01 02:17:35.008398|
|8752|Customer_ETL     |60129542146|SUCCESS|22              |2026-06-23 01:17:35.008398|2026-06-23 02:17:35.008398|
|8753|Customer_ETL     |60129542147|SUCCESS|29              |2026-06-05 01:17:35.008398|2026-06-05 02:17:35.008398|
|8754|Inventory_Sync   |60129542148|SUCCESS|15              |2026-06-13 01:17:35.008398|2026-06-13 02:17:35.008398|
|8755|Sales_Aggregation|60129542149|SUCCESS|1               |2026-06-08 